# Slide Exercise 06: Zero-Shot and Generative Recommender

This is the refined version of `Zero_Shot_Generative_Recommender.ipynb` and matches the slide exercise name `ZeroShot_Generative_Recommender.ipynb`.

Learning objectives:
- Search movies with natural-language queries.
- Use TF-IDF as a reliable zero-shot baseline.
- Optionally upgrade to SBERT embeddings.
- Treat generative metadata enrichment as a reviewed stub, not a required API call.

Main functions used:
- `vectorizer.transform(...)`: converts a new query into the same feature space as movies.
- `cosine_similarity(...)`: ranks movies against the query.
- `try/except`: enables optional semantic embeddings safely.
- Custom stub functions: show where generative enrichment could be added.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


Zero-shot search means the user can write a request directly instead of choosing a known item.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

movies["zero_shot_text"] = movies["title"] + " " + movies["genres"].str.replace("|", " ", regex=False) + " " + movies["description"] + " " + movies["keywords"]

vectorizer = TfidfVectorizer(stop_words="english")
item_matrix = vectorizer.fit_transform(movies["zero_shot_text"])

def zero_shot_search(query, n=5):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, item_matrix).ravel()
    results = movies[["title", "genres", "description"]].copy()
    results["score"] = scores
    return results.sort_values("score", ascending=False).head(n)

zero_shot_search("movies about space exploration")


Try several natural-language requests.


In [ ]:
queries = [
    "movies about space exploration",
    "light comedy for family evening",
    "romantic drama with music",
]

pd.concat(
    [zero_shot_search(q, n=3).assign(query=q) for q in queries],
    ignore_index=True,
)[["query", "title", "score", "genres"]]


Optional semantic embeddings can improve zero-shot behavior, but the exercise remains complete without them.


In [ ]:
semantic_available = False
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    semantic_item_matrix = model.encode(movies["zero_shot_text"].tolist(), show_progress_bar=False)
    semantic_available = True
except Exception as exc:
    print("Optional SBERT model is not available. Continue with TF-IDF zero-shot search.")
    print(type(exc).__name__, str(exc)[:160])

semantic_available


Keep generative enrichment as a safe, reviewed placeholder.


In [ ]:
def generative_metadata_enrichment_stub(title, short_description):
    return {
        "title": title,
        "suggested_tags": ["review-before-use", "course-demo", "generated-metadata-placeholder"],
        "draft_description": short_description,
        "warning": "Generated metadata should be reviewed before it changes recommendations.",
    }

generative_metadata_enrichment_stub(
    "Example New Movie",
    "A crew searches for a safe planet after Earth becomes difficult to inhabit.",
)


Interpretation:

Zero-shot recommendation is useful for cold-start discovery and natural-language search. Generative metadata can help fill gaps, but it must be reviewed because generated content can be wrong or inconsistent.

Student task:
1. Write a query that should retrieve `The Matrix`.
2. Add a new movie with sparse metadata and test whether the query search can find it.
